# EmotionCLIP on SEED-VII — Main Pipeline (Kaggle)

复现 **"Cross-domain EEG-based Emotion Recognition with Contrastive Learning" (Yan et al., 2025)**。

本 Notebook 只做**编排**：它把放在 Kaggle Dataset 里的脚本（`config.py` / `data_seedvii.py` / `sst_legovit.py` / `text_tower.py` / `train_emotionclip.py`）加入 `sys.path` 后直接调用，**不在 Notebook 内重写任何核心逻辑**。

### 期望的 Kaggle 输入结构
```
/kaggle/input/<你的代码dataset>/        # 把 5 个 .py + text_protocol_template.csv 打包成一个 dataset
    config.py  data_seedvii.py  sst_legovit.py  text_tower.py  train_emotionclip.py
/kaggle/input/<你的数据dataset>/         # SEED-VII 数据
    EEG_features/   subject_1.mat ... subject_20.mat   (含 de_1..de_80)
    save_info/      *_save_info.csv
    Emotion2text/   text_protocol_template.csv
```
> 数据路径会自动发现；若失败可在 **Cell 4** 手动覆盖。

## 1. 依赖
Kaggle 通常已自带 torch / numpy / scipy / pillow。一般只需补 `einops` 与 `transformers`。若联网受限，请把 `transformers` 与 CLIP 权重也做成离线 dataset。

In [ ]:
import sys, subprocess

def _pip(pkg):
    try:
        __import__(pkg.split('==')[0].replace('-', '_'))
        print(f'[ok] {pkg} already available')
    except Exception:
        print(f'[install] {pkg}')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

for _p in ['einops', 'transformers']:
    _pip(_p)

## 2. 把 Kaggle Dataset 里的脚本加入 sys.path
自动在 `/kaggle/input` 下查找包含 `train_emotionclip.py` 的目录。

In [ ]:
import os, sys
from pathlib import Path

# 如果你知道确切路径，可直接写在这里（优先生效）：
CODE_DIR = None  # 例如 '/kaggle/input/emotionclip-code'

if CODE_DIR is None:
    search_roots = [Path('/kaggle/input'), Path('.')]
    for root in search_roots:
        if not root.exists():
            continue
        hits = list(root.rglob('train_emotionclip.py'))
        if hits:
            CODE_DIR = str(hits[0].parent)
            break

assert CODE_DIR is not None, '未找到 train_emotionclip.py，请把脚本打包成 Kaggle Dataset 或设置 CODE_DIR'
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
print('CODE_DIR =', CODE_DIR)
print('scripts  =', sorted(p.name for p in Path(CODE_DIR).glob('*.py')))

## 3. 导入脚本中的函数（核心逻辑全部来自 dataset 脚本）

In [ ]:
import importlib
import config as cfg_mod
import data_seedvii
import sst_legovit
import text_tower
import train_emotionclip

# 若你在 Kaggle 里更新了脚本 dataset，可取消下面注释强制重载
# for m in (cfg_mod, data_seedvii, sst_legovit, text_tower, train_emotionclip):
#     importlib.reload(m)

from config import get_config
from train_emotionclip import run_loso
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())

## 4. 配置
默认 SEED-VII 七类，只跑首个 LOSO fold。数据路径默认会自动发现；如需手动指定，取消注释。

In [ ]:
cfg = get_config(num_classes=7)

# ---- 如需手动覆盖数据路径（自动发现失败时）----
# cfg['data']['data_root']     = '/kaggle/input/seed-vii-kaggle/EEG_features'
# cfg['data']['saveinfo_dir']  = '/kaggle/input/seed-vii-kaggle/save_info'
# cfg['data']['text_csv_path'] = '/kaggle/input/seed-vii-kaggle/Emotion2text/text_protocol_template.csv'

# ---- 常用开关 ----
cfg['runtime']['work_dir']      = '/kaggle/working/emotionclip_ckpt'
cfg['runtime']['run_all_folds'] = False   # True = 跑全部被试 LOSO
cfg['runtime']['resume']        = True    # 断点续训
cfg['runtime']['max_train_hours'] = 8.8   # Kaggle 时长预算
cfg['solver']['num_epochs']     = 100
cfg['solver']['train_batch_size'] = 64
cfg['data']['workers']          = 2
cfg['data']['use_l2_extra_prompts'] = True  # 启用 CSV 的 L2 trial 文本作额外 prompt

import pprint; pprint.pprint(cfg)

## 5. （可选）CSV 文本协议自检
确认放在数据 dataset 里的 `text_protocol_template.csv` 能被正确解析。

In [ ]:
from config import discover_data_paths
from data_seedvii import load_two_level_texts_from_csv, EMOTION_NAMES

_cfg = discover_data_paths(cfg)
_csv = _cfg['data']['text_csv_path']
print('resolved text_csv:', _csv)
if _csv and os.path.exists(_csv):
    l1, l2 = load_two_level_texts_from_csv(_csv, _cfg['data']['n_trials'])
    print('L1 情绪键:', list(l1.keys()), '| 覆盖7类:', set(l1.keys()) == set(EMOTION_NAMES))
    print('L2 trial 条数:', len(l2))
    print('示例 trial 1:', l2.get(1, '')[:80])
else:
    print('未找到 CSV：将只用 16 个内置 prompt 模板（不影响训练）')

## 6. 训练 + 跨被试 LOSO 评估
调用 dataset 脚本里的 `run_loso`。会自动加载冻结 CLIP 文本塔、构建 prompt ensemble、训练 SST-LegoViT、按被试 LOSO 评估，并把 checkpoint / history / 结果写到 `work_dir`。

In [ ]:
results = run_loso(cfg)
results

## 7. 查看结果与训练曲线

In [ ]:
import json, glob
work_dir = cfg['runtime']['work_dir']
print('outputs in', work_dir, ':')
for f in sorted(glob.glob(os.path.join(work_dir, '*'))):
    print('  ', os.path.basename(f))

hist_files = sorted(glob.glob(os.path.join(work_dir, 'history_*.json')))
if hist_files:
    with open(hist_files[0]) as f:
        hist = json.load(f)
    try:
        import matplotlib.pyplot as plt
        ep = [r['epoch'] for r in hist]
        plt.figure(figsize=(9,4))
        plt.subplot(1,2,1); plt.plot(ep,[r['train_loss'] for r in hist]); plt.title('train_loss'); plt.xlabel('epoch')
        plt.subplot(1,2,2)
        plt.plot(ep,[r['train_acc'] for r in hist], label='train')
        plt.plot(ep,[r.get('val_acc',float('nan')) for r in hist], label='val')
        plt.plot(ep,[r['test_acc'] for r in hist], label='test')
        plt.title('accuracy'); plt.xlabel('epoch'); plt.legend()
        plt.tight_layout(); plt.show()
    except Exception as e:
        print('plot skipped:', e)
        print('last epoch row:', hist[-1])